# Searching the NBS catalogue with CSW using Python

This notebook is a practical guide to searching the Norwegian National Ground Segment for Satellite Data (NBS) catalogue through its **Catalogue Service for the Web (CSW)** interface.

You will learn how to search for products based on a number of metadata field. We will then inspect the records returned and their access links.

**Important distinction:** CSW searches catalogue metadata. It does not process satellite data, and a successful catalogue search does not guarantee that a particular access service is available for every result.

## Before you start

Install the libraries if they are not already available:

```bash
python -m pip install owslib
```


## 1. Connect to the NBS CSW service

The endpoint is the service address, rather than the catalogue's human-facing website.

In [1]:
from owslib.csw import CatalogueServiceWeb
from owslib import fes

CSW_ENDPOINT = "https://nbs.csw-staging.met.no/csw"
csw = CatalogueServiceWeb(CSW_ENDPOINT, timeout=60)

print("CSW version:", csw.version)
print("Catalogue:", csw.identification.title)


CSW version: 2.0.2
Catalogue: Norwegian National Ground Segment for Satellite Data Catalogue Interface


## 2. Make a broad text search

Start with one simple criterion. `apiso:AnyText` searches across indexed text in the metadata record. The `*` wildcard means “any sequence of characters”.

> **Note**: These queries are very broad and a larger number of records match the filters applied. However, only a small number of records are returned in each response. This is because the results are returned in 'pages'. Pagination must be used to process all records (see next section).

In [2]:
text_filter = fes.PropertyIsLike(
    propertyname="apiso:AnyText",
    literal="S2*",
    wildCard="*"
)

csw.getrecords2(
    constraints=[text_filter],
)

print("Total matches:", csw.results["matches"])
print("Returned in this response:", csw.results["returned"])

for record_id, record in csw.records.items():
    print(record_id, "—", record.title)


Total matches: 6536355
Returned in this response: 10
0000004c-4f0d-42b4-96e5-fbb03c29dbdd — S2B_MSIL2A_20250514T131709_N0511_R124_T33XVH_20250514T170950
000002df-c549-404c-b02f-716a4f9b5cfd — S2A_MSIL1C_20210709T100031_N0500_R122_T35WNS_20230129T032520
00000611-4f76-473b-8d45-9985e2ad4179 — S2A_MSIL1C_20240822T100021_N0511_R122_T34VFQ_20240822T134143
00000913-0d2c-4a07-9185-73bed35da923 — S2A_MSIL2A_20160628T114652_N0500_R023_T35XMH_20231007T213418
00000a01-f027-4cec-8b11-8f890a10e61c — S2B_MSIL1C_20240425T133719_N0510_R067_T38XMR_20240425T172239
00000de2-3fc1-46a3-a1e9-f7c21bba5290 — S2B_MSIL1C_20240820T132719_N0511_R024_T33XXH_20240820T171508
000015c8-3bda-4db2-9a84-f254e9122595 — S2B_MSIL2A_20260425T115639_N0512_R066_T33XXE_20260425T154231
00001749-c002-43e4-b89a-ecc3747c68e9 — S2B_MSIL1C_20230730T115649_N0509_R066_T39XWJ_20230730T140335
000019e2-1a8d-411d-917a-55c10c6a1517 — S2A_MSIL2A_20170807T141951_N0500_R096_T25WDQ_20230830T041309
00001bff-a725-4b14-97cd-b0235809949c — S2B_MSIL

### Search the title rather than all metadata

When you know a product naming convention, a title search can be more focused than `AnyText`. The following is an example pattern, not a guarantee that every NBS title uses exactly this naming convention.

In [3]:
title_filter = fes.PropertyIsLike(
    propertyname="apiso:Title",
    literal="*GRD*",
    wildCard="*"
)

csw.getrecords2(
    constraints=[title_filter], 
    maxrecords=10
)

print("Total title matches:", csw.results["matches"])

for record_id, record in csw.records.items():
    print(record_id, "—", record.title)


Total title matches: 405265
0000260d-d622-4667-97ca-ef504fb044af — S1C_IW_GRDH_1SDV_20260625T041547_20260625T041612_008257_010533_B862
00002d2b-2089-5e7d-b962-0f4f459a3956 — S1A_EW_GRDM_1SDH_20221019T041553_20221019T041640_045506_0570E4_D964
00006af6-7986-56c3-be0b-020e93c56d91 — S1B_IW_GRDH_1SDV_20180606T175836_20180606T175901_011256_014A85_9B1A
000070d4-f0ba-57f5-acaf-0e5276c6fbf5 — S1A_EW_GRDM_1SDH_20200905T082310_20200905T082333_034221_03F9EE_E82A
0000b409-6d9c-5574-a5ee-66460b60baa5 — S1A_IW_GRDH_1SDV_20210114T060141_20210114T060206_036130_043C59_25FE
0000babf-7f64-5b2d-8f4b-456d603b22ec — S1B_IW_GRDH_1SDV_20200223T151012_20200223T151037_020398_026A52_C6C5
0000ea70-10d2-57fa-b73e-ae847c5bc875 — S1A_IW_GRDH_1SDV_20151013T154302_20151013T154327_008136_00B6B4_DE75
00011833-bad6-5010-b560-9c153e0e8e57 — S1B_IW_GRDH_1SDV_20190812T162242_20190812T162307_017555_021043_6B14
00013bb1-3324-5a87-acb8-65ef640a12cd — S1A_EW_GRDM_1SDH_20211201T053401_20211201T053505_040811_04D83F_67C9
0001473c-

## 3. Pagination

Pagination must be used to retrieve all the pages of records returned by a query. This is done by repeatedly calling csw.getrecords2() and specifying the position from which to start retrieving records using the startposition parameter.

The CSW service returns the position of the next page in csw.results["nextrecord"]. This value can be used to update startposition for the next request.

CSW record positions are one-based, meaning that the first record has position 1. Pagination continues until csw.results["nextrecord"] returns 0, indicating that there are no more records to retrieve.

For example, the following code retrieves records in batches of 10 until all matching records have been returned. In the example below, we are forcing the code to terminate after 3 pages:


In [4]:
title_filter = fes.PropertyIsLike(
    propertyname="apiso:Title",
    literal="*GRD*",
    wildCard="*"
)

start_position = 1
page_size = 10
page = 0

while start_position and page < 3:
    page = page + 1    
    csw.getrecords2(
        constraints=[title_filter],
        maxrecords=page_size,
        startposition=start_position
    )

    for record_id, record in csw.records.items():
        print(record_id, "—", record.title)

    start_position = csw.results["nextrecord"]

0000260d-d622-4667-97ca-ef504fb044af — S1C_IW_GRDH_1SDV_20260625T041547_20260625T041612_008257_010533_B862
00002d2b-2089-5e7d-b962-0f4f459a3956 — S1A_EW_GRDM_1SDH_20221019T041553_20221019T041640_045506_0570E4_D964
00006af6-7986-56c3-be0b-020e93c56d91 — S1B_IW_GRDH_1SDV_20180606T175836_20180606T175901_011256_014A85_9B1A
000070d4-f0ba-57f5-acaf-0e5276c6fbf5 — S1A_EW_GRDM_1SDH_20200905T082310_20200905T082333_034221_03F9EE_E82A
0000b409-6d9c-5574-a5ee-66460b60baa5 — S1A_IW_GRDH_1SDV_20210114T060141_20210114T060206_036130_043C59_25FE
0000babf-7f64-5b2d-8f4b-456d603b22ec — S1B_IW_GRDH_1SDV_20200223T151012_20200223T151037_020398_026A52_C6C5
0000ea70-10d2-57fa-b73e-ae847c5bc875 — S1A_IW_GRDH_1SDV_20151013T154302_20151013T154327_008136_00B6B4_DE75
00011833-bad6-5010-b560-9c153e0e8e57 — S1B_IW_GRDH_1SDV_20190812T162242_20190812T162307_017555_021043_6B14
00013bb1-3324-5a87-acb8-65ef640a12cd — S1A_EW_GRDM_1SDH_20211201T053401_20211201T053505_040811_04D83F_67C9
0001473c-1d8e-5aaa-a574-49ac7691b406 

## 4. Filter by acquisition time

NBS advertises `apiso:TempExtent_begin` and `apiso:TempExtent_end`. These describe the temporal extent of the resource, rather than when you happen to run the query.

For an interval-overlap search, ask for records that:

1. start on or before the end of the requested period; and
2. end on or after the start of the requested period.

This includes a product whose time extent overlaps the requested period, even if it began earlier or ends later.

In [5]:
from datetime import datetime, timezone

start = datetime(2025, 6, 1, tzinfo=timezone.utc)
stop = datetime(2025, 7, 1, tzinfo=timezone.utc)

time_filters = [
    fes.PropertyIsLessThanOrEqualTo(
        propertyname="apiso:TempExtent_begin",
        literal=stop.strftime("%Y-%m-%dT%H:%M:%SZ"),
    ),
    fes.PropertyIsGreaterThanOrEqualTo(
        propertyname="apiso:TempExtent_end",
        literal=start.strftime("%Y-%m-%dT%H:%M:%SZ"),
    ),
]

csw.getrecords2(
    constraints=[fes.And(time_filters)],
    maxrecords=10,
    esn="full",
)
print("Time-overlap matches:", csw.results["matches"])
for record_id, record in csw.records.items():
    print(record_id, "—", record.title)

Time-overlap matches: 165611
00003a52-a71f-43bb-a951-0948cf8a1597 — S2A_MSIL2A_20250628T141751_N0511_R096_T33XWK_20250628T183613
00004edb-5741-3351-8448-05d01ba961a9 — S3B_SY_2_AOD____20250608T145503_20250608T153930_20250609T203514_2667_107_239______ESA_O_NT_002
00007549-0263-483b-9819-adcd74718f12 — S1A_IW_RAW__0SDV_20250605T073328_20250605T073401_059508_076336_2B73
000102fa-a86f-4abe-93dc-f4db601beb7b — S2C_MSIL2A_20250608T113651_N0511_R123_T35XMG_20250608T143908
00013399-e618-3b5a-89b1-1a4538822507 — S3B_SY_2_VGP____20250628T175813_20250628T184237_20250629T234427_2664_108_141______ESA_O_NT_002
0001c3b5-562e-4764-91ca-b5a91ac39c3d — S3A_SL_2_LST____20250611T191918_20250611T192218_20250613T065240_0179_127_042_1260_PS1_O_NT_004
00027840-3a89-4c88-ae5b-65a92377cc67 — S2C_MSIL2A_20250627T084621_N0511_R107_T37VDL_20250627T143100
00048b65-0bf6-3479-bd88-a328aa827e1d — S3B_SL_1_RBT____20250623T151024_20250623T151324_20250627T123906_0180_108_068_1440_ESA_O_NT_004
0004a620-ded1-4e53-b5c3-a60d

If you need records fully contained within the period instead, the logic is different: the record must start at or after `start` and end at or before `stop`. Be explicit about which temporal question your analysis requires.


## 5. Filter by geographic area

The service advertises `apiso:BoundingBox` and the CSW filter capabilities include the spatial `BBOX` operator.

The coordinate order used below is **west, south, east, north**, in decimal degrees. The example is a small area in northern Norway. Adjust it to your area of interest.

In [6]:
# [west, south, east, north] in longitude/latitude degrees
bbox = [18.0, 68.0, 20.0, 69.0]

bbox_filter = fes.BBox(
    bbox,
    crs="urn:ogc:def:crs:OGC:1.3:CRS84",
)

csw.getrecords2(
    constraints=[bbox_filter],
)
print("Spatial matches:", csw.results["matches"])
for record_id, record in csw.records.items():
    print(record_id, "—", record.title)

Spatial matches: 1597026
000004f5-2037-4bdb-90eb-c78e96484b80 — S3B_SL_2_WST____20241219T162421_20241219T180520_20241220T222239_6059_101_111______MAR_O_NT_003
00001415-beae-5355-a9de-314f7a170d07 — S3A_SL_2_LST____20170713T130524_20170713T144623_20180930T013944_6059_020_024______LR1_R_NT_003
00001ba1-22f2-4340-a72f-f4b7b4d41449 — S3A_SL_2_LST____20180321T230836_20180321T231136_20180322T011618_0179_029_144_1800_SVL_O_NR_002
00002332-e70b-451b-a2ea-8a7117c327c7 — S3B_SY_2_AOD____20260421T075555_20260421T084012_20260422T152138_2657_119_135______ESA_O_NT_003
00002ae9-c3f9-4133-bdec-1447e839b34d — S5P_RPRO_L2__NP_BD3_20191011T100304_20191011T114433_10329_03_010300_20220811T151935
00002e35-f2c9-5427-9bda-adb80e484a07 — S3A_SR_1_SRA____20180112T154233_20180112T163301_20180302T004129_3028_026_325______LR1_R_NT_003
000034cf-89f1-4881-8c22-804340ad80eb — S3A_SL_2_AOD____20240804T102046_20240804T102544_20240804T123924_0300_115_222______MAR_O_NR_003
0000403d-1d2a-4ce3-81de-78f2a887c2ec — S3A_OL_2_

A bounding-box search asks for spatial intersection with the rectangle. It does not mean that the product footprint is identical to the rectangle. The CRS identifier and axis-order conventions can be a source of confusion in OGC services; if results look implausible, test with a well-known area and check the service's returned bounding boxes.


## 6. Combining filters

The real value of CSW is combining independent criteria. This example combines:

- a product-name pattern in the title;
- an acquisition-time overlap;
- a geographic bounding box.

The platform value and product-title convention are examples. If they do not match your catalogue records, inspect the records and adjust them.

In [7]:
# Free text filter based on title
product_filter = fes.PropertyIsLike(
    propertyname="apiso:Title",
    literal="S2*L2A*",
    wildCard="*"
)

# [west, south, east, north] in longitude/latitude degrees
bbox = [18.0, 68.0, 20.0, 69.0]

bbox_filter = fes.BBox(
    bbox,
    crs="urn:ogc:def:crs:OGC:1.3:CRS84",
)

# Time filter
start = datetime(2026, 6, 1, tzinfo=timezone.utc)
stop = datetime(2026, 9, 1, tzinfo=timezone.utc)

time_filters = [
    fes.PropertyIsLessThanOrEqualTo(
        propertyname="apiso:TempExtent_begin",
        literal=stop.strftime("%Y-%m-%dT%H:%M:%SZ"),
    ),
    fes.PropertyIsGreaterThanOrEqualTo(
        propertyname="apiso:TempExtent_end",
        literal=start.strftime("%Y-%m-%dT%H:%M:%SZ"),
    ),
]

# Combining filters
combined_filter = fes.And([
    product_filter,
    *time_filters,
    bbox_filter,
])

sort_order = fes.SortBy([
    fes.SortProperty("apiso:TempExtent_begin", "ASC")
])

csw.getrecords2(
    constraints=[combined_filter],
    sortby=sort_order,
)

print("Total matching records:", csw.results["matches"])
for record_id, record in csw.records.items():
    print(record_id, "—", record.title)


Total matching records: 457
8de1fa54-c0d9-4bc3-937f-6884d51f8d71 — S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417
946dea87-e5aa-4776-ba4a-942ef2be1041 — S2A_MSIL2A_20260601T102041_N0512_R065_T34WDA_20260601T164417
5a7dbf97-55a3-49c4-8f2e-8cac54f27b6c — S2C_MSIL2A_20260601T105621_N0512_R094_T34WDB_20260601T141912
7c9d6630-72bd-4bf2-969b-75d37bcc6c7f — S2C_MSIL2A_20260601T105621_N0512_R094_T33WXR_20260601T141912
95222cc6-ab3a-46ee-af1d-538a844dd67d — S2C_MSIL2A_20260601T105621_N0512_R094_T33WXS_20260601T141912
64482c6b-9034-4bae-8180-20778dc58cce — S2C_MSIL2A_20260602T103021_N0512_R108_T33WXS_20260602T155116
db4b1835-ac72-47c3-b4b1-42b60dc0bd71 — S2C_MSIL2A_20260602T103021_N0512_R108_T34WDA_20260602T155116
f4abdb1b-b404-41a1-8524-b7fa0e8532c0 — S2C_MSIL2A_20260602T103021_N0512_R108_T33WXR_20260602T155116
fdfbdbfe-6397-412b-a33e-b3125f9748f9 — S2C_MSIL2A_20260602T103021_N0512_R108_T34WDB_20260602T155116
3b2a1bd0-6829-4e3d-888a-af62dc774be0 — S2B_MSIL2A_20260603T104619_N0512_

## 7. Inspect a record

For a specific record, inspect its references. A reference is a link or service endpoint associated with the metadata record; different schemes represent different access methods.

In [8]:
if csw.records:
    first_id, first_record = next(iter(csw.records.items()))
    print("Record:", first_id)
    print("Title:", first_record.title)
    print("Abstract:", first_record.abstract)
    print("\nReferences:")
    for reference in first_record.references:
        print(reference)
else:
    print("No records were collected. Broaden or adjust the search.")


Record: 8de1fa54-c0d9-4bc3-937f-6884d51f8d71
Title: S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417
Abstract: Level-2A ortho-rectified and UTM geo-coded Bottom-of-Atmosphere multi-spectral reflectance from the Sentinel 2 Multi-Spectral Imager. Additional outputs are Aerosol Optical Thickness (AOT) map, Water Vapour (WV) map, Scene Classification map together with Quality Indicators data.

References:
{'scheme': 'OPeNDAP:OPeNDAP', 'url': 'https://nbstds.met.no/thredds/dodsC/NBS/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417.nc'}
{'scheme': 'OGC:WMS', 'url': 'https://adc-wms.met.no/get_wms/8de1fa54-c0d9-4bc3-937f-6884d51f8d71/wms'}
{'scheme': 'WWW:DOWNLOAD-1.0-http--download', 'url': 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417.zip'}
{'scheme': 'WWW:DOWNLOAD-1.0-http--download', 'url': 'https://nbstds.met.no/thredds/fileServer/NBS/S2A/2026/06/01/S2A_MSIL2A_2026060

## 8. Retrieving the data access links

Here we retrieve the various data access links for each record and write them to a dictionary. The following access is provided depending on the product:

- **Direct download** is generally for obtaining a local copy of a file.
- **OPeNDAP** can support remote access to data where the service and data format allow it.
- **WMS** provides rendered map imagery for visualisation; it is not a substitute for the original measurement data.

### All access URLs in a dictionary

In [9]:
for key, value in list(csw.records.items()):
    for ref in value.references:
        print(ref)

{'scheme': 'OPeNDAP:OPeNDAP', 'url': 'https://nbstds.met.no/thredds/dodsC/NBS/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417.nc'}
{'scheme': 'OGC:WMS', 'url': 'https://adc-wms.met.no/get_wms/8de1fa54-c0d9-4bc3-937f-6884d51f8d71/wms'}
{'scheme': 'WWW:DOWNLOAD-1.0-http--download', 'url': 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417.zip'}
{'scheme': 'WWW:DOWNLOAD-1.0-http--download', 'url': 'https://nbstds.met.no/thredds/fileServer/NBS/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417.nc'}
{'scheme': 'OPeNDAP:OPeNDAP', 'url': 'https://nbstds.met.no/thredds/dodsC/NBS/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T34WDA_20260601T164417.nc'}
{'scheme': 'OGC:WMS', 'url': 'https://adc-wms.met.no/get_wms/946dea87-e5aa-4776-ba4a-942ef2be1041/wms'}
{'scheme': 'WWW:DOWNLOAD-1.0-http--download', 'url': 'https://nbstds.met.no/thredds/fileServer/nbsArchive/

### Returning a List of OPeNDAP URLs

In [10]:
url_opendap = []

for key, value in list(csw.records.items()):
    for ref in value.references:
        if ref['scheme'] == 'OPeNDAP:OPeNDAP':
            url_opendap.append(ref['url'])

url_opendap

['https://nbstds.met.no/thredds/dodsC/NBS/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417.nc',
 'https://nbstds.met.no/thredds/dodsC/NBS/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T34WDA_20260601T164417.nc',
 'https://nbstds.met.no/thredds/dodsC/NBS/S2C/2026/06/01/S2C_MSIL2A_20260601T105621_N0512_R094_T34WDB_20260601T141912.nc',
 'https://nbstds.met.no/thredds/dodsC/NBS/S2C/2026/06/01/S2C_MSIL2A_20260601T105621_N0512_R094_T33WXR_20260601T141912.nc',
 'https://nbstds.met.no/thredds/dodsC/NBS/S2C/2026/06/01/S2C_MSIL2A_20260601T105621_N0512_R094_T33WXS_20260601T141912.nc',
 'https://nbstds.met.no/thredds/dodsC/NBS/S2C/2026/06/02/S2C_MSIL2A_20260602T103021_N0512_R108_T33WXS_20260602T155116.nc',
 'https://nbstds.met.no/thredds/dodsC/NBS/S2C/2026/06/02/S2C_MSIL2A_20260602T103021_N0512_R108_T34WDA_20260602T155116.nc',
 'https://nbstds.met.no/thredds/dodsC/NBS/S2C/2026/06/02/S2C_MSIL2A_20260602T103021_N0512_R108_T33WXR_20260602T155116.nc',
 'https://nbstds

### Returning a List of WMS URLs

In [11]:
url_wms = []

for key, value in list(csw.records.items()):
    for ref in value.references:
        if ref['scheme'] == 'OGC:WMS':
            url_wms.append(ref['url'])

url_wms

['https://adc-wms.met.no/get_wms/8de1fa54-c0d9-4bc3-937f-6884d51f8d71/wms',
 'https://adc-wms.met.no/get_wms/946dea87-e5aa-4776-ba4a-942ef2be1041/wms',
 'https://adc-wms.met.no/get_wms/5a7dbf97-55a3-49c4-8f2e-8cac54f27b6c/wms',
 'https://adc-wms.met.no/get_wms/7c9d6630-72bd-4bf2-969b-75d37bcc6c7f/wms',
 'https://adc-wms.met.no/get_wms/95222cc6-ab3a-46ee-af1d-538a844dd67d/wms',
 'https://adc-wms.met.no/get_wms/64482c6b-9034-4bae-8180-20778dc58cce/wms',
 'https://adc-wms.met.no/get_wms/db4b1835-ac72-47c3-b4b1-42b60dc0bd71/wms',
 'https://adc-wms.met.no/get_wms/f4abdb1b-b404-41a1-8524-b7fa0e8532c0/wms',
 'https://adc-wms.met.no/get_wms/fdfbdbfe-6397-412b-a33e-b3125f9748f9/wms',
 'https://adc-wms.met.no/get_wms/3b2a1bd0-6829-4e3d-888a-af62dc774be0/wms']

### Returning a List of URLs for Direct Download

In [12]:
url_download = []

for key, value in list(csw.records.items()):
    for ref in value.references:
        if ref['scheme'] == 'WWW:DOWNLOAD-1.0-http--download':
            url_download.append(ref['url'])

url_download

['https://nbstds.met.no/thredds/fileServer/nbsArchive/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417.zip',
 'https://nbstds.met.no/thredds/fileServer/NBS/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T33WXR_20260601T164417.nc',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T34WDA_20260601T164417.zip',
 'https://nbstds.met.no/thredds/fileServer/NBS/S2A/2026/06/01/S2A_MSIL2A_20260601T102041_N0512_R065_T34WDA_20260601T164417.nc',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2C/2026/06/01/S2C_MSIL2A_20260601T105621_N0512_R094_T34WDB_20260601T141912.zip',
 'https://nbstds.met.no/thredds/fileServer/NBS/S2C/2026/06/01/S2C_MSIL2A_20260601T105621_N0512_R094_T34WDB_20260601T141912.nc',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2C/2026/06/01/S2C_MSIL2A_20260601T105621_N0512_R094_T33WXR_20260601T141912.zip',
 'https://nbstds.met.no/thredds/fileServer/NBS/S2C/2026/06/01/S2C_MSIL2A

## 9. Full example of Querying and Returning URLs for Direct Download

In [13]:
from owslib.csw import CatalogueServiceWeb
from owslib import fes
from datetime import datetime, timezone

CSW_ENDPOINT = "https://nbs.csw-staging.met.no/csw"
csw = CatalogueServiceWeb(CSW_ENDPOINT, timeout=60)

# Free text filter based on title
product_filter = fes.PropertyIsLike(
    propertyname="apiso:Title",
    literal="S2*L2A*",
    wildCard="*"
)

# [west, south, east, north] in longitude/latitude degrees
bbox = [18.0, 68.0, 20.0, 69.0]

bbox_filter = fes.BBox(
    bbox,
    crs="urn:ogc:def:crs:OGC:1.3:CRS84",
)

# Time filter
start = datetime(2025, 6, 1, tzinfo=timezone.utc)
stop = datetime(2025, 9, 1, tzinfo=timezone.utc)

time_filters = [
    fes.PropertyIsLessThanOrEqualTo(
        propertyname="apiso:TempExtent_begin",
        literal=stop.strftime("%Y-%m-%dT%H:%M:%SZ"),
    ),
    fes.PropertyIsGreaterThanOrEqualTo(
        propertyname="apiso:TempExtent_end",
        literal=start.strftime("%Y-%m-%dT%H:%M:%SZ"),
    ),
]

# Combining filters
combined_filter = fes.And([
    product_filter,
    *time_filters,
    bbox_filter,
])

sort_order = fes.SortBy([
    fes.SortProperty("apiso:TempExtent_begin", "ASC")
])

start_position = 1
page_size = 10
page = 0
url_download = []

while start_position and page < 3:
    page = page + 1    
    csw.getrecords2(
        constraints=[combined_filter],
        maxrecords=page_size,
        startposition=start_position
    )

    for key, value in list(csw.records.items()):
        for ref in value.references:
            if ref['scheme'] == 'WWW:DOWNLOAD-1.0-http--download':
                url_download.append(ref['url'])

    start_position = csw.results["nextrecord"]

url_download

['https://nbstds.met.no/thredds/fileServer/nbsArchive/S2C/2025/07/06/S2C_MSIL2A_20250706T105641_N0511_R094_T33WXS_20250706T145919.zip',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2A/2025/07/25/S2A_MSIL2A_20250725T105051_N0511_R051_T33WXS_20250725T122714.zip',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2B/2025/08/07/S2B_MSIL2A_20250807T104619_N0511_R051_T33WXR_20250807T114648.zip',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2C/2025/07/23/S2C_MSIL2A_20250723T105051_N0511_R051_T34WDB_20250723T142813.zip',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2A/2025/08/08/S2A_MSIL2A_20250808T102701_N0511_R108_T34WDA_20250808T123121.zip',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2A/2025/08/31/S2A_MSIL2A_20250831T103701_N0511_R008_T33WXR_20250831T130614.zip',
 'https://nbstds.met.no/thredds/fileServer/nbsArchive/S2B/2025/08/11/S2B_MSIL2A_20250811T102559_N0511_R108_T33WXS_20250811T124556.zip',
 'https://nbstds.met.no/thredds/fileServer/nbsAr

## 10. Troubleshooting and interpretation

- **Connection or timeout error:** this is a service/network issue, not a zero-result search.
- **Zero matches:** check the field name and exact value, then broaden one condition at a time.
- **Fewer records returned than matches:** the response is paginated or limited by `maxrecords`.
- **Unexpected time results:** confirm UTC handling and whether you want temporal overlap or full containment.
- **Unexpected spatial results:** check coordinate order, CRS and axis-order conventions.
